In [1]:
import pandas as pd
import json
import nltk
import re
import html
#nltk.download('punkt')
#nltk.download("punkt_tab")

In [2]:
tweets_df = pd.read_pickle("data/tweets_full_new.pkl")
senate_votes = pd.read_csv("data/senate_votes_members.csv")
house_votes = pd.read_csv("data/house_votes_members.csv")

In [3]:
# --- URLs -------------------------------------------------------------------
# The old pattern was r'https://\S+', which missed plain http:// and bare
# t.co/ links — 12.6% of the corpus still contained "http://t.co/..." after
# cleaning, and those tokens show up as junk terms in any topic model.
link_pattern = re.compile(r'(?:https?://|www\.)\S+|\bt\.co/\S+', re.IGNORECASE)

# --- Hashtags and mentions ---------------------------------------------------
# The old pattern was r'[#@]\w+', which deleted the whole token *including the
# word*: "#immigration" vanished entirely, leaving a whitespace hole. Hashtags
# are the highest-signal topic tokens in political Twitter, so we keep the word
# and drop only the symbol.
#
# camelCase is split so multi-word hashtags become real words:
#   #BuildTheWall -> "Build The Wall"      #MAGA -> "MAGA"
#   @RepJoeSmith  -> "Rep Joe Smith"
#
# IMPORTANT: this must run BEFORE .lower(). Once the text is lowercased the
# word boundaries inside a hashtag are gone and camelCase can never be
# recovered.
hashtag_pattern = re.compile(r'[#@](\w+)')

_camel_pattern = re.compile(r'(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])')

def strip_tag(match: re.Match) -> str:
    """Drop the leading # or @, keep the word, split camelCase."""
    return " " + _camel_pattern.sub(" ", match.group(1))

# Collapse the whitespace left behind by removals.
whitespace_pattern = re.compile(r'\s+')

# emoji_pattern = re.compile("["
#                                u"\U0001F600-\U0001F64F"  # emoticons
#                                u"\U0001F300-\U0001F5FF"  # symbols & pictographs
#                                u"\U0001F680-\U0001F6FF"  # transport & map symbols
#                                u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
#                                u"\U00002702-\U000027B0"
#                                u"\U000024C2-\U0001F251"
#                                "]+", flags=re.UNICODE)

In [4]:
# Order matters. Hashtags/mentions are processed BEFORE lowercasing so that
# camelCase can still be split; the old version lowercased first, which would
# make "#BuildTheWall" -> "#buildthewall" and lose the word boundaries.
# Links are stripped first so that a "#" inside a URL is never mistaken for a
# hashtag.

text = tweets_df["text"].astype(str)

# 1. newlines -> spaces
text = text.str.replace(r"\r\n|\r|\n", " ", regex=True)

# 1b. HTML entities (&amp; -> &, &lt; -> <, &#39; -> ', etc.) — Twitter's API
# returns entity-escaped text; left undecoded, "&amp;" survives as the
# literal token "amp" and pollutes downstream topic modelling.
text = text.map(html.unescape)

# 2. URLs (http, https, www, bare t.co)
text = text.str.replace(link_pattern, " ", regex=True)

# 3. hashtags/mentions: keep the word, drop the symbol, split camelCase
#    (still original case at this point)
text = text.str.replace(hashtag_pattern, strip_tag, regex=True)

# 4. now it is safe to lowercase
text = text.str.lower()

# 5. collapse the whitespace left by the removals above
text = text.str.replace(whitespace_pattern, " ", regex=True).str.strip()

tweets_df["text"] = text
tweets_df = tweets_df[tweets_df["text"] != ""]

print(f"{len(tweets_df):,} tweets after cleaning")
print(f"  still containing 'http': {tweets_df['text'].str.contains('http').mean()*100:.2f}%")
print(f"  still containing '#'/'@': {tweets_df['text'].str.contains(r'[#@]').mean()*100:.2f}%")
tweets_df["text"].head(5).tolist()

4,974,695 tweets after cleaning
  still containing 'http': 0.00%
  still containing '#'/'@': 0.43%


['violent crime continues to climb all across the country, the southern border continues being overrun with illegals, deadly fentanyl and other dangerous drugs are pouring in, and as we saw just in the last few days, america is seen as weak by our adversaries…especially china.',
 'cpl. fred mcgee sr., a native of bloomingdale, heroically served our nation in the korean war and risked his life to aid his squad amid battle. while he ordered his squad to withdraw from battle, he stayed behind to evacuate wounded and deceased soldiers.',
 'my legislation in honor of spc christian ward was passed to ensure no family member will bear this burden again.',
 'military families whose family members have made the ultimate sacrifice shouldn’t be burdened with worrying about whether or not they will receive their loved one’s possessions.',
 'potus – hardworking americans who have seen their paychecks and 401(k) plans vanish because of bidenflation don’t think the economy is “strong as hell.”']

In [5]:
keep_questions = [
    "On Passage of the Bill",
    "On Passage",
    "On Overriding the Veto",
    "On the Cloture Motion",
    "On Agreeing to the Conference Report",
    "On the Conference Report",
    "On Motion to Suspend the Rules and Pass",
    "On the Joint Resolution",
    "On Consideration of the Joint Resolution",
]

senate_votes = senate_votes[senate_votes["question"].isin(keep_questions)]
house_votes = house_votes[house_votes["question"].isin(keep_questions)]

In [6]:
tweets_df.to_pickle("data/tweets_full_new.pkl")
senate_votes.to_csv("data/senate_votes_members.csv")
house_votes.to_csv("data/house_votes_members.csv")